## Nexora Care Flow – Project.
1. Company Overview
Nexora Care is a healthcare technology company focused on supporting small and mid-sized clinics. Its platform, Nexora Care Flow, integrates appointment scheduling, patient records, telemedicine, care coordination, staffing, and analytics into a single system. The company aims to replace fragmented healthcare software with a simpler, connected platform that reduces administrative workload and improves operational efficiency.

2. Project Overview
The Nexora Care Flow Appointment Volume Forecasting Project uses historical appointment data from pilot clinics to predict future appointment demand. The project applies time-series analysis to identify patterns such as day-of-week effects, holidays, and seasonal demand. The resulting forecasts will help clinic administrators anticipate workload and make better staffing and room-allocation decisions.

3. Business Problem
Nexora's pilot clinics currently plan staffing and room capacity largely through gut feeling, historical averages, and reactive decision-making. Because appointment demand can fluctuate significantly, administrators may not know that a busy period is coming until demand has already increased. This can result in understaffing during busy periods and overstaffing during quieter periods.

4. Business Impact
The lack of accurate demand visibility creates several business impacts:
•	Higher labour costs from unnecessary staffing during quiet periods. 
•	Staff overload and overtime during unexpected demand surges. 
•	Longer patient waiting times when clinics are understaffed. 
•	Reduced operational efficiency across reception, clinical, telemedicine, and care-coordination workflows. 
•	Poor resource allocation, particularly for staff and consultation rooms. 
•	Reduced patient experience and satisfaction during periods of high demand. 
Ultimately, the problem undermines Nexora's goal of reducing operational friction for clinics.
5. Project Objectives

The project has five main objectives:
1 – Establish Forecast Accuracy Benchmarks
Measure the forecasting model's performance against a naive baseline using MAPE and RMSE.
2 – Identify Demand Drivers
Analyse historical appointment data to identify day-of-week, holiday, seasonal, and clinic-specific demand patterns.
3 – Build a Volume Forecasting Model
Develop a time-series forecasting model capable of predicting future appointment volumes at clinic level over a multi-week horizon.
4 – Translate Forecasts into Actionable Guidance
Convert predicted appointment volumes into practical staffing and capacity guidance that clinic administrators can use for forward planning.
5 – Establish a Path for Continuous Improvement
Document a process for comparing forecasts with actual appointment volumes and periodically refining the model as more data becomes available.
In conclusion, Nexora Care Flow is a healthcare technology platform designed to simplify operations for small and mid-sized clinics by integrating scheduling, patient records, telemedicine, staffing, and analytics. This project addresses the problem of unpredictable appointment demand by using historical scheduling data to identify demand patterns and develop a time-series forecasting approach. By providing reliable clinic-level forecasts, the project aims to help administrators plan staffing and resources proactively, reducing labour inefficiencies, staff pressure, patient waiting times, and operational disruption while establishing a foundation for future data-driven decision-making.


In [1]:
# Import Libraries
import pandas as pd
import numpy as nb

In [ ]:
# Load the raw appointment records CSV file into a pandas DataFrame
df = pd.read_csv("C:/Users/telvi/Downloads/AMDARI/Nexora_Care_Flow/data/raw_data/AppointmentRecords.csv")

In [ ]:
# Display the first five rows to inspect the dataset structure and values
df.head()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
0,529385,2,Nexora Care - Lakeside,Multi-Specialty,103642,202,2024-01-01 08:00,New Patient,Completed,Online,2,True,New Year's Day,Flu Season,False
1,500000,1,Nexora Care - Riverside,Primary Care,101551,101,2024-01-01 08:30,New Patient,No-Show,Online,18,True,New Year's Day,Flu Season,False
2,500011,1,Nexora Care - Riverside,Primary Care,100260,104,2024-01-01 08:30,Follow-Up,Completed,Referral,4,True,New Year's Day,Flu Season,False
3,575066,3,Nexora Care - Downtown Express,Telehealth-Forward,104923,301,2024-01-01 09:45,Urgent,Completed,Online,0,True,New Year's Day,Flu Season,False
4,500008,1,Nexora Care - Riverside,Primary Care,101124,104,2024-01-01 09:45,Telehealth,Completed,Phone,2,True,New Year's Day,Flu Season,False


In [ ]:
# Generate summary statistics for all numerical and categorical columns
df.describe(include='all')

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
count,129353.000000,129353.000000,129353,129353,129353.000000,129353.000000,129353,129353,129353,129353,129353.000000,129353,939,129353,129353
unique,NaN,NaN,4,3,NaN,NaN,30287,4,4,4,NaN,2,13,3,2
top,NaN,NaN,Nexora Care - Lakeside,Primary Care,NaN,NaN,2025-12-30 11:15,Follow-Up,Completed,Online,NaN,False,Christmas Eve,Standard,False
freq,NaN,NaN,45681,58526,NaN,NaN,19,57859,109040,57061,NaN,128414,105,64015,92038
mean,564676.000000,2.417864,NaN,NaN,103553.485385,245.290005,NaN,NaN,NaN,NaN,8.704267,NaN,NaN,NaN,NaN
std,37341.139023,1.071525,NaN,NaN,2020.941152,107.165883,NaN,NaN,NaN,NaN,7.641921,NaN,NaN,NaN,NaN
min,500000.000000,1.000000,NaN,NaN,100000.000000,101.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
25%,532338.000000,2.000000,NaN,NaN,101812.000000,201.000000,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN,NaN
50%,564676.000000,2.000000,NaN,NaN,103547.000000,205.000000,NaN,NaN,NaN,NaN,7.000000,NaN,NaN,NaN,NaN
75%,597014.000000,3.000000,NaN,NaN,105178.000000,306.000000,NaN,NaN,NaN,NaN,12.000000,NaN,NaN,NaN,NaN


In [ ]:
# Display information about the DataFrame, including columns, data types, and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           939 non-null     object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
dtypes: bool(2), int64

In [6]:
# Check the number of missing (null) values in each column
df.isnull().sum()

AppointmentID                0
ClinicID                     0
ClinicName                   0
ClinicType                   0
PatientID                    0
ProviderID                   0
AppointmentDateTime          0
AppointmentType              0
Status                       0
BookingChannel               0
BookingLeadTimeDays          0
IsHoliday                    0
HolidayName             128414
SeasonFlag                   0
ChronicConditionFlag         0
dtype: int64

In [ ]:
## Count the number of unique values in each column to identify 
# categorical variables and potential data quality issues
df.nunique()

AppointmentID           129353
ClinicID                     4
ClinicName                   4
ClinicType                   3
PatientID                 7109
ProviderID                  24
AppointmentDateTime      30287
AppointmentType              4
Status                       4
BookingChannel               4
BookingLeadTimeDays         61
IsHoliday                    2
HolidayName                 14
SeasonFlag                   3
ChronicConditionFlag         2
dtype: int64

In [7]:
# Check whether the IsHoliday flag is consistent with the presence or absence of a HolidayName
pd.crosstab(df['IsHoliday'], df['HolidayName'].isna())

HolidayName,False,True
IsHoliday,,
False,0,128414
True,939,0


In [8]:
# AppointmentDateTime needs to be treated as a datetime variable
df['AppointmentDateTime'] = pd.to_datetime(df['AppointmentDateTime'])

In [9]:
#verify if data type of the AppointmentDateTime column has changed to datetype 
df['AppointmentDateTime'].dtype

dtype('<M8[ns]')

In [26]:
df.isnull().sum()

AppointmentID           0
ClinicID                0
ClinicName              0
ClinicType              0
PatientID               0
ProviderID              0
AppointmentDateTime     0
AppointmentType         0
Status                  0
BookingChannel          0
BookingLeadTimeDays     0
IsHoliday               0
HolidayName             0
SeasonFlag              0
ChronicConditionFlag    0
dtype: int64

In [10]:
# Count missing values in AppointmentDateTime
df['AppointmentDateTime'].isnull().sum()

np.int64(0)

In [11]:
# Replace missing holiday names with "No Holiday"
# to clearly distinguish non-holiday records from missing data.
df['HolidayName'] = df['HolidayName'].fillna('No Holiday')

In [12]:
# Check whether any missing values remain in the HolidayName column
df['HolidayName'].isnull().sum()

np.int64(0)

In [13]:
# Count the number of records for each holiday name
# to understand the distribution of holidays and non-holiday records.
df['HolidayName'].value_counts()

HolidayName
No Holiday                128414
Christmas Eve                105
Thanksgiving                  87
New Year's Eve                87
Christmas Day                 80
New Year's Day                74
Presidents' Day               73
Juneteenth                    72
Day After Thanksgiving        64
Independence Day              63
MLK Day                       59
Veterans Day                  59
Memorial Day                  58
Labor Day                     58
Name: count, dtype: int64

In [14]:
# Remove or flag duplicate records
# Check the number of duplicate records in the dataset
df.duplicated().sum()

np.int64(0)

In [15]:
# Identify records with negative booking lead times,
# which are logically invalid because an appointment, cannot be booked before the booking date.
df[df['BookingLeadTimeDays'] < 0]

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag


In [16]:
# Identifier Validation (Appointment, clinic, patient and provider IDs were checked for invalid non-positive values:)
df[
    (df['AppointmentID'] <= 0) |
    (df['ClinicID'] <= 0) |
    (df['PatientID'] <= 0) |
    (df['ProviderID'] <= 0)
]



,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag


In [17]:
# Appointment Status (The Status field was examined)
# to identify the distribution of valid and cancelled appointments
df['Status'].value_counts()

Status
Completed      109040
No-Show         11367
Cancelled        6296
Rescheduled      2650
Name: count, dtype: int64

In [18]:
# Whitespace and formatting inconsistencies
# Remove leading and trailing spaces and standardise the
df['Status'] = df['Status'].str.strip()
# capitalisation of appointment status values for consistency.
df['Status'] = df['Status'].str.title()

In [19]:
# Clinic Volume and Closure Anomalies
# Calculate the total number of appointments for each clinic
# by grouping records by ClinicID and ClinicName.
clinic_volume = (
    df.groupby(['ClinicID', 'ClinicName'])
      .size()
      .reset_index(name='AppointmentCount')
)

clinic_volume.sort_values(
    'AppointmentCount',
    ascending=True
).head(20)


,ClinicID,ClinicName,AppointmentCount
2,3,Nexora Care - Downtown Express,25146
3,4,Nexora Care - Hillcrest,29144
0,1,Nexora Care - Riverside,29382
1,2,Nexora Care - Lakeside,45681


In [20]:
# save to my processed_data folder
df.to_csv(
    r'C:\Users\telvi\Downloads\AMDARI\Nexora_Care_Flow\data\processed_data\AppointmentRecords_Cleaned.csv',
    index=False
)